# Sentiment analysis 

## 1. Textblob-FR

Documentation: https://textblob.readthedocs.io/en/dev/

### Imports

In [3]:

from pathlib import Path
import os, re, csv

CWD = Path.cwd()
candidates = []
for p in [CWD, *CWD.parents]:
    cand = p / "data" / "txt" / "1950"
    if cand.exists():
        candidates.append(cand)
TXT_DIR = candidates[0] if candidates else None
assert TXT_DIR and TXT_DIR.exists(), f"Dossier 1950 introuvable depuis {CWD}. Cherché sous */data/txt/1950."

print(f"[OK] Dossier textes: {TXT_DIR}")

POS = {
    "excellent","excellente","excellentes","excellents","remarquable","remarquables",
    "bonne","bon","bons","bonnes","superbe","magnifique","favorable","positif","positive",
    "succès","remerciements","félicitations","bravo","heureux","avantage","avantages",
    "progrès","amélioration","meilleur","meilleure","meilleures","meilleurs","gagner",
    "victoire","talent","prestige","qualité","fiable","utile","agréable","confort","économie",
    "réussite","record","gratuit","promotion","bénéfice","bénéfices","satisfaction"
}
NEG = {
    "mauvais","mauvaise","mauvaises","mediocre","médiocre","faible","faibles",
    "défaut","défauts","risque","risques","danger","dangers","perte","pertes","crise","crises",
    "difficile","difficiles","problème","problèmes","retard","retards","erreur","erreurs",
    "plainte","plaintes","dommage","dommages","manque","pénurie","coût","coûts",
    "interdiction","interdit","dégradation","baisse","chute","défaite","défaites","échec","échecs",
    "conflit","conflits","grève","grèves","accident","accidents"
}
tok = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ'-]{3,}")

def score_text(text:str):
    words = [w.lower() for w in tok.findall(text)]
    pos = sum(1 for w in words if w in POS)
    neg = sum(1 for w in words if w in NEG)
    total = len(words)
    pol = (pos - neg) / (pos + neg + 1e-9)  
    return pos, neg, total, pol

OUT_DIR = (CWD.parent / "output" / "sentiment"); OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = (CWD.parent / "figs"); FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV   = OUT_DIR / "sentiment_1950.csv"
TOP_POS   = OUT_DIR / "sentiment_1950_top_pos.csv"
TOP_NEG   = OUT_DIR / "sentiment_1950_top_neg.csv"
PNG_HIST  = FIG_DIR / "sentiment_hist_1950.png"

files = sorted([f for f in os.listdir(TXT_DIR) if f.endswith(".txt")])
files = files[:500] if len(files) > 500 else files
print(f"[OK] Fichiers à traiter: {len(files)}")

rows = []
for f in files:
    text = (TXT_DIR / f).read_text(encoding="utf-8", errors="ignore")
    pos, neg, n, pol = score_text(text)
    rows.append([f, pos, neg, n, pol])


with OUT_CSV.open("w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh)
    w.writerow(["doc_id","pos","neg","n_tokens","polarity"])
    w.writerows(rows)


rows_sorted = sorted(rows, key=lambda r: r[-1])               
with TOP_NEG.open("w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh); w.writerow(["doc_id","pos","neg","n_tokens","polarity"])
    w.writerows(rows_sorted[:20])
with TOP_POS.open("w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh); w.writerow(["doc_id","pos","neg","n_tokens","polarity"])
    w.writerows(list(reversed(rows_sorted[-20:])))

print(f"[OK] CSV -> {OUT_CSV}\n[OK] TOP+ -> {TOP_POS}\n[OK] TOP- -> {TOP_NEG}")


try:
    import matplotlib.pyplot as plt
    vals = [r[-1] for r in rows]
    plt.figure(figsize=(9,5))
    plt.hist(vals, bins=40)
    plt.title("Répartition des polarités – 1950 (lexique simple)")
    plt.xlabel("polarité"); plt.ylabel("documents")
    plt.tight_layout(); plt.savefig(PNG_HIST, dpi=150); plt.close()
    print(f"[OK] FIG -> {PNG_HIST}")
except Exception as e:
    print("[INFO] Matplotlib indisponible, pas de figure:", e)


[OK] Dossier textes: /Users/iduoliveira/TACTEST/tac/data/txt/1950
[OK] Fichiers à traiter: 100
[OK] CSV -> /Users/iduoliveira/TACTEST/tac/tp2/output/sentiment/sentiment_1950.csv
[OK] TOP+ -> /Users/iduoliveira/TACTEST/tac/tp2/output/sentiment/sentiment_1950_top_pos.csv
[OK] TOP- -> /Users/iduoliveira/TACTEST/tac/tp2/output/sentiment/sentiment_1950_top_neg.csv
[OK] FIG -> /Users/iduoliveira/TACTEST/tac/tp2/figs/sentiment_hist_1950.png


In [1]:
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

### Création d'une fonction `get_sentiment`

In [4]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def get_sentiment(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    print(f"This text is {polarity_str} and {subjectivity_str}.")

### Analyser le sentiment d'une phrase

In [ ]:
get_sentiment("Ce journal est vraiment super intéressant.")

In [ ]:
get_sentiment("Cette phrase est négative et je ne suis pas content !")

## 2. Utilisation de transformers

Documentation: https://github.com/TheophileBlard/french-sentiment-analysis-with-bert

**!!** Si le code ne tourne pas sur votre machine, vous pouvez le tester directement sur Google Colab en utilisant [ce lien](https://colab.research.google.com/github/TheophileBlard/french-sentiment-analysis-with-bert/blob/master/colab/french_sentiment_analysis_with_bert.ipynb) **!!**

Le modèle peut également être testé en ligne sur [HuggingFace](https://huggingface.co/tblard/tf-allocine)

### Installation des librairies et imports

In [ ]:
%pip install tensorflow
%pip install sentencepiece
%pip install transformers
%pip install tf_keras

from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from transformers import pipeline

### Chargement du modèle

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("tblard/tf-allocine", use_pt=True)
model = TFAutoModelForSequenceClassification.from_pretrained("tblard/tf-allocine")

sentiment_analyser = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

### Analyser le sentiment d'une phrase

In [39]:
sentiment_analyser("Ce journal est vraiment super intéressant.")

In [ ]:
sentiment_analyser("Cette phrase est négative et je ne suis pas content !")